# 🐟 Fish Speech — Voice Cloning Finetune v4
### `mih12345/may_30_english` · torch 2.8.0 · CUDA 12.8 · Python 3.12

**Before running:** Runtime → Change runtime type → **A100 GPU** + **High RAM**

Run cells top to bottom. Do not restart runtime between cells.

## Cell 0 — Check Environment

In [ ]:
!nvidia-smi
import sys, psutil, torch
print(f'Python:  {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'RAM:     {psutil.virtual_memory().total/1e9:.0f} GB')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Cell 1 — Install Dependencies
Pins torch 2.8.0+cu128 and installs all required packages.

In [ ]:
import os

# System
!apt-get install -q -y portaudio19-dev ffmpeg

# Clone fish-speech
if not os.path.exists('/content/fish-speech'):
    !git clone https://github.com/fishaudio/fish-speech.git /content/fish-speech
else:
    !cd /content/fish-speech && git pull --quiet
print('Repo ready.')

%cd /content/fish-speech

# Pin torch 2.8.0 + cu128 FIRST — must happen before anything else
!pip install -q torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 \
    --index-url https://download.pytorch.org/whl/cu128

# Install fish-speech (v2.0.0 has no [train] extra)
!pip install -q -e .

# Training deps
!pip install -q lightning pytorch-lightning
!pip install -q 'einx[torch]==0.2.2'

# Audio + data
!pip install -q pyloudnorm soundfile librosa
!pip install -q datasets huggingface_hub
!pip install -q 'protobuf>=3.20,<4.0'
!pip install -q tiktoken
!pip install -q pyaudio

# Verify
import importlib
for pkg in ['torch','torchaudio','fish_speech','hydra','lightning','tiktoken','pyloudnorm']:
    try:
        m = importlib.import_module(pkg)
        print(f'OK  {pkg} {getattr(m, "__version__", "")}' )
    except ImportError as e:
        print(f'MISSING  {pkg}  {e}')

## Cell 2 — Download Base Model (openaudio-s1-mini)

In [ ]:
from huggingface_hub import snapshot_download
import os

CHECKPOINT_DIR = '/content/fish-speech/checkpoints/openaudio-s1-mini'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(os.path.join(CHECKPOINT_DIR, 'model.pth')):
    print('Downloading openaudio-s1-mini (~3.6 GB)...')
    snapshot_download(
        repo_id='fishaudio/openaudio-s1-mini',
        local_dir=CHECKPOINT_DIR,
        repo_type='model',
        ignore_patterns=['*.md', '*.txt']
    )
else:
    print('Already downloaded.')

for f in ['model.pth', 'codec.pth', 'tokenizer.tiktoken', 'special_tokens.json', 'config.json']:
    p = os.path.join(CHECKPOINT_DIR, f)
    print(f'{f}: {"OK" if os.path.exists(p) else "MISSING"}')

## Cell 3 — Convert Dataset → WAV + TXT
Handles `dict` and `AudioDecoder` (torchcodec) formats automatically.
Saves as `.txt` (not `.lab`) — required by `build_dataset.py`.

In [ ]:
import os, numpy as np, soundfile as sf, librosa
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm

SPEAKER_NAME = 'SPK_MAY30'
DATA_DIR     = Path('/content/fish-speech/data') / SPEAKER_NAME
TARGET_SR    = 44100
MIN_DURATION = 1.0
MAX_DURATION = 15.0
DATA_DIR.mkdir(parents=True, exist_ok=True)

existing = list(DATA_DIR.glob('*.wav'))
if len(existing) >= 400:
    print(f'Already converted: {len(existing)} files. Skipping.')
else:
    print('Loading dataset...')
    dataset = load_dataset('mih12345/may_30_english', split='train')
    print(f'Samples: {len(dataset)}')

    s0 = dataset[0]['audio']
    print(f'Audio type: {type(s0)}')

    saved = skipped = 0
    errors = []

    for idx, sample in enumerate(tqdm(dataset, desc='Converting')):
        text = sample['text'].strip()
        if not text:
            skipped += 1
            continue
        try:
            audio = sample['audio']
            if isinstance(audio, dict):
                arr = np.array(audio['array'], dtype=np.float32)
                sr  = audio['sampling_rate']
            elif hasattr(audio, 'get_all_samples'):
                s   = audio.get_all_samples()
                arr = s.data.numpy().astype(np.float32)
                sr  = s.sample_rate
            else:
                raise ValueError(f'Unknown audio type: {type(audio)}')

            if arr.ndim > 1:
                arr = arr.mean(axis=0)
            arr = arr.astype(np.float32)

            dur = len(arr) / sr
            if not (MIN_DURATION <= dur <= MAX_DURATION):
                skipped += 1
                continue

            if sr != TARGET_SR:
                arr = librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR)

            arr = np.clip(arr, -1.0, 1.0)
            sf.write(str(DATA_DIR / f'{idx:06d}.wav'), arr, TARGET_SR, subtype='PCM_16')
            (DATA_DIR / f'{idx:06d}.txt').write_text(text, encoding='utf-8')
            saved += 1
        except Exception as e:
            errors.append((idx, str(e)))
            skipped += 1

    print(f'Saved: {saved}  Skipped: {skipped}  Errors: {len(errors)}')
    if errors:
        print('First 3 errors:', errors[:3])

wav_count = len(list(DATA_DIR.glob('*.wav')))
txt_count = len(list(DATA_DIR.glob('*.txt')))
print(f'WAV: {wav_count}  TXT: {txt_count}')
assert wav_count == txt_count, 'WAV/TXT count mismatch!'

## Cell 4 — Loudness Normalization (-23 LUFS)

In [ ]:
import soundfile as sf
import pyloudnorm as pyln
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path('/content/fish-speech/data/SPK_MAY30')
wav_files = sorted(DATA_DIR.glob('*.wav'))
print(f'Normalizing {len(wav_files)} files to -23 LUFS...')

meter_cache = {}
skipped = 0
for wav_path in tqdm(wav_files):
    try:
        data, sr = sf.read(str(wav_path))
        if sr not in meter_cache:
            meter_cache[sr] = pyln.Meter(sr)
        loudness = meter_cache[sr].integrated_loudness(data)
        if loudness < -70:
            skipped += 1
            continue
        data = np.clip(pyln.normalize.loudness(data, loudness, -23.0), -1.0, 1.0)
        sf.write(str(wav_path), data, sr, subtype='PCM_16')
    except Exception:
        skipped += 1

print(f'Done. Skipped {skipped} files.')

## Cell 5 — Extract VQ Tokens (.npy)
Bypasses `extract_vq.py` entirely (incompatible with torch 2.8).
Calls codec `.encode()` directly. Saves as `[n_codebooks, T]` int32.

In [ ]:
from pathlib import Path
import numpy as np

DATA_DIR = Path('/content/fish-speech/data/SPK_MAY30')
npy_existing = list(DATA_DIR.glob('*.npy'))

if len(npy_existing) >= 400:
    print(f'Already extracted: {len(npy_existing)} NPY files. Skipping.')
else:
    script = r"""
import sys
sys.path.insert(0, "/content/fish-speech")
import torch, soundfile as sf, numpy as np
from pathlib import Path
from tqdm import tqdm
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

DATA_DIR   = Path("/content/fish-speech/data/SPK_MAY30")
CODEC_PATH = "/content/fish-speech/checkpoints/openaudio-s1-mini/codec.pth"
CONFIG_DIR = "/content/fish-speech/fish_speech/configs"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

with initialize_config_dir(config_dir=CONFIG_DIR, version_base=None):
    cfg = compose(config_name="modded_dac_vq")

model = instantiate(cfg)
model.load_state_dict(torch.load(CODEC_PATH, map_location="cpu"), strict=False)
model = model.to(DEVICE).eval()
print(f"Codec loaded on {DEVICE}")

wav_files = sorted(DATA_DIR.glob("*.wav"))
print(f"Processing {len(wav_files)} files...")
errors = 0

for wav_path in tqdm(wav_files):
    npy_path = wav_path.with_suffix(".npy")
    if npy_path.exists():
        continue
    try:
        data, sr = sf.read(str(wav_path))
        if data.ndim > 1:
            data = data.mean(axis=1)
        x = torch.tensor(data, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            codes, _ = model.encode(x)   # [1, n_codebooks, T]
        arr = codes.cpu().numpy().squeeze(0).astype(np.int32)  # [n_codebooks, T]
        np.save(str(npy_path), arr)
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"Error {wav_path.name}: {e}")

npy = list(DATA_DIR.glob("*.npy"))
print(f"Done. NPY: {len(npy)}  Errors: {errors}")
sample = np.load(str(npy[0]))
print(f"Shape: {sample.shape}  dtype: {sample.dtype}")
"""
    with open('/content/extract_vq_custom.py', 'w') as f:
        f.write(script)
    !cd /content/fish-speech && python /content/extract_vq_custom.py

npy = list(DATA_DIR.glob('*.npy'))
wav = list(DATA_DIR.glob('*.wav'))
print(f'NPY: {len(npy)}  WAV: {len(wav)}')
if len(npy) != len(wav):
    print('WARNING: count mismatch')

## Cell 6 — Pack into .protos

In [ ]:
import os
from pathlib import Path

PROTOS_DIR = '/content/fish-speech/data/protos'
os.makedirs(PROTOS_DIR, exist_ok=True)

# Ensure correct protobuf version for fish-speech protos
!pip install -q 'protobuf>=3.20,<4.0'

existing_protos = list(Path(PROTOS_DIR).glob('*.protos'))
if existing_protos:
    print(f'Already packed: {len(existing_protos)} proto file(s).')
    for pf in existing_protos:
        print(f'  {pf.name}: {pf.stat().st_size/1e6:.1f} MB')
else:
    !cd /content/fish-speech && python tools/llama/build_dataset.py \
        --input /content/fish-speech/data/SPK_MAY30 \
        --output /content/fish-speech/data/protos \
        --num-workers 1

    proto_files = list(Path(PROTOS_DIR).glob('*.protos'))
    print(f'Proto files: {len(proto_files)}')
    for pf in proto_files:
        print(f'  {pf.name}: {pf.stat().st_size/1e6:.1f} MB')
    assert proto_files, 'No .protos files created — check errors above'

## Cell 7 — Apply All Patches

**Three patches applied here:**
1. **tokenizer.py** — replaces `AutoTokenizer` with native tiktoken loader
2. **text2semantic_finetune.yaml** — hardcodes proto path, fixes tokenizer path
3. **train.py** — manually instantiates datamodule to avoid Hydra DictConfig errors

In [ ]:
from pathlib import Path
import re

# ══════════════════════════════════════════════════════════════
# PATCH 1 — Replace tokenizer.py with native tiktoken loader
# ══════════════════════════════════════════════════════════════
TOKENIZER_PY = r'''
import json
import base64
import logging
from pathlib import Path
from typing import List

import tiktoken

logger = logging.getLogger(__name__)

SEMANTIC_TOKEN_TEMPLATE = "<|semantic:{i}|>"
SEMANTIC_TOKENS = [SEMANTIC_TOKEN_TEMPLATE.format(i=i) for i in range(4096)]

MODALITY_TOKENS = {
    "text": "<|text|>",
    "voice": "<|voice|>",
    "interleave": "<|interleave|>",
}

ALL_SPECIAL_TOKENS = [
    "<|endoftext|>", "<|pad|>", "<|im_start|>", "<|im_end|>",
    "<|phoneme_start|>", "<|phoneme_end|>",
    "<|text|>", "<|voice|>", "<|interleave|>",
    "<|audio_start|>", "<|audio_end|>", "<|audio_pad|>",
    *SEMANTIC_TOKENS,
]


def _load_tiktoken_bpe(path: str) -> dict:
    """Load tiktoken .tiktoken file -> {bytes: int}."""
    ranks = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            token_b64, rank_str = line.split()
            ranks[base64.b64decode(token_b64)] = int(rank_str)
    return ranks


class FishTokenizer:
    def __init__(self, model_path: str):
        p = Path(model_path)
        tok_file  = (p / "tokenizer.tiktoken") if p.is_dir() else p
        spec_file = tok_file.parent / "special_tokens.json"

        self._special_tokens: dict = json.loads(spec_file.read_text())
        mergeable_ranks = _load_tiktoken_bpe(str(tok_file))

        self._enc = tiktoken.Encoding(
            name="fish",
            pat_str=r"""(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+""",
            mergeable_ranks=mergeable_ranks,
            special_tokens=self._special_tokens,
        )

        # Build string-keyed vocab
        self._vocab: dict = {}
        for token_bytes, rank in mergeable_ranks.items():
            try:
                self._vocab[token_bytes.decode("utf-8")] = rank
            except UnicodeDecodeError:
                self._vocab[token_bytes.decode("utf-8", errors="replace")] = rank
        self._vocab.update(self._special_tokens)

        # Semantic token mapping
        self.semantic_id_to_token_id: dict = {}
        for i in range(4096):
            tok = SEMANTIC_TOKEN_TEMPLATE.format(i=i)
            if tok in self._special_tokens:
                self.semantic_id_to_token_id[i] = self._special_tokens[tok]

        if not self.semantic_id_to_token_id:
            logger.warning("No semantic tokens found in vocabulary!")
        else:
            logger.info(f"Loaded {len(self.semantic_id_to_token_id)} semantic tokens.")

    # ── Required interface ────────────────────────────────────────────

    def __len__(self) -> int:
        return len(self._vocab)

    @property
    def vocab_size(self) -> int:
        return len(self._vocab)

    @property
    def pad_token_id(self) -> int:
        return self._special_tokens.get("<|pad|>", 0)

    @property
    def eos_token_id(self) -> int:
        return self._special_tokens.get("<|end_of_text|>", 0)

    def get_vocab(self) -> dict:
        return self._vocab

    def get_token_id(self, token: str) -> int:
        return self._vocab.get(token, 0)

    def convert_tokens_to_ids(self, token: str) -> int:
        return self._vocab.get(token, 0)

    def encode(self, text: str, **kwargs) -> List[int]:
        return self._enc.encode(text, allowed_special="all")

    def decode(self, tokens, **kwargs) -> str:
        if hasattr(tokens, "tolist"):
            tokens = tokens.tolist()
        return self._enc.decode(tokens)

    def save_pretrained(self, path):
        import shutil
        import os
        os.makedirs(path, exist_ok=True)
        src = Path(str(self._enc.name))  # fallback
        # Copy tokenizer files
        ckpt_dir = Path(path)
        for f in ["tokenizer.tiktoken", "special_tokens.json"]:
            candidate = ckpt_dir.parent / f
            if candidate.exists():
                shutil.copy2(str(candidate), os.path.join(path, f))

    def __getattr__(self, name):
        raise AttributeError(f"FishTokenizer has no attribute {name!r}")
'''

Path('/content/fish-speech/fish_speech/tokenizer.py').write_text(TOKENIZER_PY.strip())
print('PATCH 1 done: tokenizer.py')

# ══════════════════════════════════════════════════════════════
# PATCH 2 — Fix YAML config
# ══════════════════════════════════════════════════════════════
cfg_path = Path('/content/fish-speech/fish_speech/configs/text2semantic_finetune.yaml')
cfg = cfg_path.read_text()

# Fix tokenizer: point to directory not .tiktoken file
cfg = cfg.replace(
    'model_path: ${pretrained_ckpt_path}/tokenizer.tiktoken',
    'model_path: ${pretrained_ckpt_path}'
)
# Hardcode absolute proto paths (avoids Hydra list parse bug)
cfg = cfg.replace(
    'proto_files:\n    - data/protos',
    'proto_files:\n    - /content/fish-speech/data/protos'
)

cfg_path.write_text(cfg)
print('PATCH 2 done: text2semantic_finetune.yaml')

# Verify the two key lines
for line in cfg.splitlines():
    if 'model_path' in line or 'protos' in line:
        if not line.strip().startswith('#'):
            print(' ', line.strip())

# ══════════════════════════════════════════════════════════════
# PATCH 3 — Fix train.py: bypass Hydra DictConfig for datamodule
# ══════════════════════════════════════════════════════════════
!cd /content/fish-speech && git checkout -- fish_speech/train.py

train_path = Path('/content/fish-speech/fish_speech/train.py')
train_src  = train_path.read_text()

OLD = '    datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data)'

NEW = '''\
    # Manually instantiate datamodule — avoids Hydra DictConfig being passed
    # as a dataset object (causes ConfigKeyError: Missing key 0)
    from hydra.utils import instantiate as _inst
    from fish_speech.datasets.semantic import SemanticDataModule
    from fish_speech.tokenizer import FishTokenizer as _FT

    _tok    = _FT(cfg.pretrained_ckpt_path)
    _trd    = _inst(cfg.train_dataset)
    _vald   = _inst(cfg.val_dataset)

    datamodule: LightningDataModule = SemanticDataModule(
        train_dataset=_trd,
        val_dataset=_vald,
        batch_size=cfg.data.batch_size,
        tokenizer=_tok,
        max_length=cfg.max_length,
        num_workers=0,
    )'''

if OLD in train_src:
    train_path.write_text(train_src.replace(OLD, NEW))
    print('PATCH 3 done: train.py')
else:
    print('PATCH 3 WARNING: target line not found in train.py')
    print('Lines containing "datamodule":')
    for i, l in enumerate(train_src.splitlines()):
        if 'datamodule' in l.lower():
            print(f'  L{i}: {l}')

# Clear bytecode cache
!find /content/fish-speech -name '*.pyc' -delete 2>/dev/null
!find /content/fish-speech -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null
print('Bytecode cache cleared.')

# ── Smoke test tokenizer ──────────────────────────────────────
import sys, importlib
sys.path.insert(0, '/content/fish-speech')
import fish_speech.tokenizer
importlib.reload(fish_speech.tokenizer)
from fish_speech.tokenizer import FishTokenizer
tok = FishTokenizer('/content/fish-speech/checkpoints/openaudio-s1-mini')
print(f'Tokenizer OK  vocab={tok.vocab_size}  semantic={len(tok.semantic_id_to_token_id)}')
print(f'Encode test: {tok.encode("hello world")[:5]}')

## Cell 8 — Configure Training Hyperparameters

In [ ]:
import torch, re
from pathlib import Path

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram_gb:.1f} GB')

if vram_gb >= 35:       # A100 40GB
    BATCH_SIZE, GRAD_ACCUM = 4, 2
elif vram_gb >= 20:     # L4, 4090
    BATCH_SIZE, GRAD_ACCUM = 2, 4
else:                   # T4 16GB
    BATCH_SIZE, GRAD_ACCUM = 1, 8

MAX_STEPS = 2000
print(f'batch={BATCH_SIZE}  grad_accum={GRAD_ACCUM}  eff_batch={BATCH_SIZE*GRAD_ACCUM}  steps={MAX_STEPS}')

cfg_path = Path('/content/fish-speech/fish_speech/configs/text2semantic_finetune.yaml')
cfg = cfg_path.read_text()
cfg = re.sub(r'batch_size:.*',              f'batch_size: {BATCH_SIZE}',   cfg)
cfg = re.sub(r'accumulate_grad_batches:.*', f'accumulate_grad_batches: {GRAD_ACCUM}', cfg)
cfg = re.sub(r'max_steps:.*',              f'max_steps: {MAX_STEPS}',     cfg)
cfg_path.write_text(cfg)
print('Config written.')

## Cell 9 — LoRA Finetune
A100 40GB: ~40–50 min for 2000 steps. Watch `train/loss` decrease in output.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
PROJECT_NAME = 'may30_english_voice'

print(f'Starting: {PROJECT_NAME}')
print(f'Checkpoints: /content/fish-speech/results/{PROJECT_NAME}/')

!cd /content/fish-speech && python fish_speech/train.py \
    --config-name text2semantic_finetune \
    project={PROJECT_NAME} \
    pretrained_ckpt_path=/content/fish-speech/checkpoints/openaudio-s1-mini \
    +lora@model.model.lora_config=r_8_alpha_16

## Cell 10 — Merge LoRA Weights

In [ ]:
from pathlib import Path

PROJECT_NAME = 'may30_english_voice'
RESULTS_DIR  = Path(f'/content/fish-speech/results/{PROJECT_NAME}/checkpoints')
MERGED_DIR   = '/content/fish-speech/checkpoints/may30_voice_merged'

checkpoints = sorted(RESULTS_DIR.glob('step_*.ckpt'))
print(f'Checkpoints found: {len(checkpoints)}')
for c in checkpoints:
    print(f'  {c.name}  ({c.stat().st_size/1e6:.0f} MB)')

if not checkpoints:
    print('No checkpoints — did Cell 9 complete?')
else:
    SELECTED = checkpoints[-1]
    print(f'Merging: {SELECTED.name}')

    !cd /content/fish-speech && python tools/llama/merge_lora.py \
        --lora-config r_8_alpha_16 \
        --base-weight checkpoints/openaudio-s1-mini \
        --lora-weight {SELECTED} \
        --output {MERGED_DIR}

    if Path(MERGED_DIR).exists():
        print('Merged model files:')
        for f in Path(MERGED_DIR).iterdir():
            print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')
    else:
        print('Merge failed — check output above')

## Cell 11 — Test Inference

In [ ]:
import soundfile as sf, os
from pathlib import Path
from IPython.display import Audio, display

SPEAKER_NAME = 'SPK_MAY30'
MERGED_DIR   = '/content/fish-speech/checkpoints/may30_voice_merged'
DATA_DIR     = Path(f'/content/fish-speech/data/{SPEAKER_NAME}')

def get_dur(p):
    d, sr = sf.read(str(p))
    return len(d) / sr

wav_files = sorted(DATA_DIR.glob('*.wav'))
good_refs = [f for f in wav_files if 4.0 <= get_dur(f) <= 7.0] or wav_files[:5]
REF_AUDIO = good_refs[0]
REF_TEXT  = REF_AUDIO.with_suffix('.txt').read_text(encoding='utf-8').strip()

print(f'Reference: {REF_AUDIO.name}  ({get_dur(REF_AUDIO):.1f}s)')
print(f'Text: "{REF_TEXT[:80]}"')

TEST_SENTENCES = [
    "Hello, this is a test of the cloned voice. The quick brown fox jumps over the lazy dog.",
    "The sun rises in the east and sets in the west, painting the sky with brilliant colors.",
    "In machine learning, the model learns patterns from data to make predictions on unseen examples.",
    "Please confirm your appointment for tomorrow morning at nine o'clock sharp.",
]

for i, sentence in enumerate(TEST_SENTENCES):
    out_path = f'/content/fish-speech/output_test_{i+1}.wav'
    print(f'\n[{i+1}] {sentence[:70]}')
    os.system(
        f'cd /content/fish-speech && python fish_speech/inference.py '
        f'--text "{sentence}" '
        f'--prompt-text "{REF_TEXT}" '
        f'--prompt-speech {REF_AUDIO} '
        f'--checkpoint-path {MERGED_DIR} '
        f'--output {out_path} '
        f'--device cuda 2>/dev/null'
    )
    if os.path.exists(out_path):
        display(Audio(out_path))
    else:
        print('  Failed to generate')

## Cell 12 — Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

MERGED_DIR   = '/content/fish-speech/checkpoints/may30_voice_merged'
DATA_DIR     = Path('/content/fish-speech/data/SPK_MAY30')
SAVE_DIR     = Path('/content/drive/MyDrive/fish_speech_may30')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Merged model
dest = SAVE_DIR / 'may30_voice_merged'
if not dest.exists():
    print('Copying model...')
    shutil.copytree(MERGED_DIR, str(dest))
print(f'Model: {dest}')

# Reference audio + transcript
import soundfile as sf
def get_dur(p):
    d, sr = sf.read(str(p)); return len(d)/sr
wav_files = sorted(DATA_DIR.glob('*.wav'))
good_refs = [f for f in wav_files if 4.0 <= get_dur(f) <= 7.0] or wav_files[:1]
REF_AUDIO = good_refs[0]
REF_TEXT  = REF_AUDIO.with_suffix('.txt').read_text(encoding='utf-8').strip()
shutil.copy2(str(REF_AUDIO), str(SAVE_DIR / 'reference_audio.wav'))
(SAVE_DIR / 'reference_text.txt').write_text(REF_TEXT)
print('Reference audio + text saved.')

# Test outputs
import os
for wav in Path('/content/fish-speech').glob('output_test*.wav'):
    shutil.copy2(str(wav), str(SAVE_DIR / wav.name))
print(f'All saved to: {SAVE_DIR}')

---
## Troubleshooting

**`ConfigKeyError: Missing key 0`** — Re-run Cell 7. The YAML proto path replacement didn't apply.

**`FishTokenizer has no attribute 'from_pretrained'`** — Re-run Cell 7. Bytecode cache may be stale.

**Audio sounds robotic / monotone** — Try an earlier checkpoint (step 500–1000). With 454 clips (~30 min audio), overfitting can happen at 2000 steps.

**CUDA OOM** — Reduce `BATCH_SIZE` to 1 and `GRAD_ACCUM` to 8 in Cell 8.

**For FR / IT / ES voices** — Same pipeline. Swap the HF dataset in Cell 3 and change `SPEAKER_NAME`. Keep one language per speaker folder.